In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/08 22:37:16 WARN Utils: Your hostname, ZenbookUX3402ZA, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/08 22:37:16 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 22:37:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/08 22:37:30 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/08 22:37:30 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
df_green = spark.read.format("parquet").option("recursiveFileLookup","true").load('data/pq/green')

In [3]:
df_green.registerTempTable('green')

/mnt/d/DEV/DEZoocamp2026/06_batch/.venv/lib/python3.12/site-packages/pyspark/sql/classic/dataframe.py:178: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [8]:
df_green_revenue = spark.sql("""
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [9]:
df_green_revenue.show()

[Stage 7:=================>                                       (5 + 11) / 16]

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-24 09:00:00|  81|             59.49|             2|
|2020-01-04 21:00:00|  25|            513.83|            32|
|2020-01-10 19:00:00|  66| 545.6800000000002|            27|
|2020-01-30 07:00:00|  75| 556.6600000000001|            40|
|2020-01-18 01:00:00| 260|            144.56|            12|
|2020-01-12 08:00:00| 177|31.090000000000003|             2|
|2020-01-20 21:00:00| 166|            133.28|            12|
|2020-01-03 04:00:00|  14|            105.34|             2|
|2020-01-30 20:00:00|  74| 766.4400000000002|            58|
|2020-01-02 16:00:00|  66|             257.0|            12|
|2020-01-11 23:00:00|  42|            340.47|            31|
|2020-01-13 12:00:00|  89|            135.25|             6|
|2020-01-02 08:00:00| 174|50.309999999999995|             3|
|2020-01-20 01:00:00|  4

In [10]:
df_green_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/green', mode='overwrite')

26/03/08 22:42:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/08 22:42:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/08 22:42:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/08 22:42:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/08 22:42:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/08 22:42:37 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/08 22:42:37 WARN MemoryManager: Total allocation exceeds 95.

In [11]:
df_yellow = spark.read.format("parquet").option("recursiveFileLookup","true").load('data/pq/yellow')
df_yellow.registerTempTable('yellow')

In [12]:
df_yellow_revenue = spark.sql("""
SELECT 
    date_trunc('hour', tpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [13]:
df_yellow_revenue.show()

[Stage 17:=====================================================>  (20 + 1) / 21]

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-03 19:00:00| 142| 6023.089999999995|           403|
|2020-01-26 14:00:00| 239| 6541.649999999994|           437|
|2020-01-09 01:00:00| 100|            653.56|            37|
|2020-01-31 18:00:00| 237|12689.400000000009|           810|
|2020-01-04 03:00:00| 246|2092.5400000000004|           121|
|2020-01-14 09:00:00| 239| 4882.359999999997|           298|
|2020-01-27 16:00:00| 162| 7989.979999999996|           452|
|2020-01-17 20:00:00| 170| 6884.189999999997|           407|
|2020-01-23 15:00:00| 142| 5378.829999999997|           341|
|2020-01-27 06:00:00|  24|            536.49|            23|
|2020-01-01 04:00:00| 170|            2306.2|           135|
|2020-01-05 12:00:00|  68|3495.9599999999987|           235|
|2020-01-13 17:00:00| 107|4066.6799999999976|           241|
|2020-01-21 19:00:00| 16

In [14]:
df_yellow_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/yellow', mode='overwrite')

26/03/08 22:50:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/08 22:50:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/08 22:50:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/08 22:50:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/08 22:50:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/08 22:50:53 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/08 22:50:53 WARN MemoryManager: Total allocation exceeds 95.

In [15]:
df_green_revenue = spark.read.parquet('data/report/revenue/green')
df_yellow_revenue = spark.read.parquet('data/report/revenue/yellow')

In [16]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

In [17]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

In [18]:
df_join.show()

[Stage 29:==================================>                      (6 + 4) / 10]

+-------------------+----+------------------+--------------------+------------------+---------------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+----+------------------+--------------------+------------------+---------------------+
|2020-01-01 00:00:00|  55|129.29000000000002|                   4|              NULL|                 NULL|
|2020-01-01 00:00:00|  60|            160.04|                   6|57.620000000000005|                    2|
|2020-01-01 00:00:00|  61|            526.71|                  17|            146.64|                    3|
|2020-01-01 00:00:00|  65|            199.49|                  10|            409.35|                   19|
|2020-01-01 00:00:00|  71|              23.8|                   1|              NULL|                 NULL|
|2020-01-01 00:00:00|  78|             34.46|                   2|              NULL|                 NULL|
|2020-01-01 00:00:00|  88|  

In [19]:
df_join.write.parquet('data/report/revenue/total', mode='overwrite')

26/03/08 23:30:42 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/08 23:30:42 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/08 23:30:42 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/08 23:30:42 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/08 23:30:42 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/08 23:30:42 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/08 23:30:42 WARN MemoryManager: Total allocation exceeds 95.

In [20]:
df_join = spark.read.parquet('data/report/revenue/total')

In [21]:
df_join.show()

+-------------------+----+------------------+--------------------+------------------+---------------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+----+------------------+--------------------+------------------+---------------------+
|2020-01-01 00:00:00|  13|              NULL|                NULL|1214.8000000000002|                   56|
|2020-01-01 00:00:00|  48|              NULL|                NULL|10773.360000000004|                  455|
|2020-01-01 00:00:00|  76|143.77999999999997|                   4|             35.51|                    2|
|2020-01-01 00:00:00| 130|            133.35|                   7|              NULL|                 NULL|
|2020-01-01 00:00:00| 186|              NULL|                NULL| 4011.449999999998|                  188|
|2020-01-01 00:00:00| 208|             80.24|                   3|              NULL|                 NULL|
|2020-01-01 00:00:00| 248|  

In [22]:
df_zones = spark.read.parquet('zones/')

In [23]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [24]:
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

In [25]:
df_result.show()

+-------------------+----+------------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|LocationID|  Borough|                Zone|service_zone|
+-------------------+----+------------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|2020-01-01 00:00:00|  13|              NULL|                NULL|1214.8000000000002|                   56|        13|Manhattan|   Battery Park City| Yellow Zone|
|2020-01-01 00:00:00|  48|              NULL|                NULL|10773.360000000004|                  455|        48|Manhattan|        Clinton East| Yellow Zone|
|2020-01-01 00:00:00|  76|143.77999999999997|                   4|             35.51|                    2|        76| Brooklyn|       East New York|   Boro Zone|
|2020-01-01 00:00:00| 

In [26]:
df_result.drop('LocationID', 'zone').write.parquet('tmp/revenue-zones')

26/03/08 23:47:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/08 23:47:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/08 23:47:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/08 23:47:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/08 23:47:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/08 23:47:33 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/08 23:47:33 WARN MemoryManager: Total allocation exceeds 95.